# M4 · Métricas de transcripción


## 1. Clonar repositorio

In [1]:
RAMA = "Luis"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo

Cloning into '/content/repo'...
remote: Enumerating objects: 1658, done.
remote: Counting objects: 100% (384/384), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 1658 (delta 230), reused 253 (delta 185), pack-reused 1274 (from 1)
Receiving objects: 100% (1658/1658), 33.46 MiB | 28.96 MiB/s, done.
Resolving deltas: 100% (760/760), done.


In [2]:
%cd /content/repo
!git pull
%cd M4/metricas_transcripcion
!pip install -r requirements.txt -q

/content/repo
Already up to date.
/content/repo/M4/metricas_transcripcion


## 2. Montar Drive y definir `PROJECT_ROOT`

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

manifest = PROJECT_ROOT / "M4/data/imagenes/manifest.json"
print("Manifiesto:", manifest, "| existe:", manifest.exists())
print("Caches de transcripcion disponibles:")
for p in sorted((PROJECT_ROOT / "M4/vision/predictions").glob("transcripciones_*.json")):
    print("  ", p.relative_to(PROJECT_ROOT))

Mounted at /content/drive
Manifiesto: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json | existe: True
Caches de transcripcion disponibles:
   M4/vision/predictions/transcripciones_vlm_foto_realista.json
   M4/vision/predictions/transcripciones_vlm_limpio.json
   M4/vision/predictions/transcripciones_vlm_perspectiva_severo.json
   M4/vision/predictions/transcripciones_vlm_rotacion_recorte_severo.json
   M4/vision/predictions/transcripciones_vlm_sombra_severo.json


## 3. Probar el módulo con datos simulados (opcional)

Genera transcripciones con errores conocidos y comprueba que las métricas los miden. **No son resultados del sistema**: van a `M4/outputs/metricas_transcripcion/simulado/`.

In [4]:
!python run_metricas.py --simular

[03:26:13] Gold: M4/eval_set/eval_set_casos.jsonl | 15 casos, 57 entidades
[03:26:13] Cache NER del sistema sobre texto: M2/predictions/encoder_solo.json
[03:26:17] Simulacion: 6 niveles en /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches
[03:26:17] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_limpio.json
[03:26:17] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_foto_realista.json
[03:26:17] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_manuscrito_foto.json
[03:26:18] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_sombra_severo.json
[03:26:18] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Sa

## 4. Medir las transcripciones reales

Un cache de transcripciones por nivel (y, si existe, su cache NER en el mismo orden). Las rutas son relativas a `PROJECT_ROOT`. También se pueden fijar en `entradas` de `config.yaml`.

In [5]:
TRANSCRIPCIONES = [
    "M4/vision/predictions/transcripciones_vlm_limpio.json",
    "M4/vision/predictions/transcripciones_vlm_foto_realista.json",
]
NER = [
     "M2/predictions/vision_vlm_limpio.json",
     "M2/predictions/vision_vlm_foto_realista.json",
]

args = " ".join(f'"{t}"' for t in TRANSCRIPCIONES)
args_ner = (" --ner " + " ".join(f'"{n}"' for n in NER)) if NER else ""
if TRANSCRIPCIONES:
    !python run_metricas.py --transcripciones {args}{args_ner}
else:
    print("Agrega al menos un cache en TRANSCRIPCIONES")

[03:26:20] Gold: M4/eval_set/eval_set_casos.jsonl | 15 casos, 57 entidades
[03:26:20] Cache NER del sistema sobre texto: M2/predictions/encoder_solo.json
[03:26:20] Evaluando M4/vision/predictions/transcripciones_vlm_limpio.json
[03:26:21] Evaluando M4/vision/predictions/transcripciones_vlm_foto_realista.json
[03:26:22] Listo en 2.4 s: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion
vlm          limpio                       CER_vis=0.0002 WER_vis=0.0014 lectura=1.0 inventadas=0.0
vlm          foto_realista                CER_vis=0.0029 WER_vis=0.0041 lectura=1.0 inventadas=0.0


## 5. Ver el resumen

In [6]:
from IPython.display import Markdown, display

for sub in ("", "simulado"):
    ruta = PROJECT_ROOT / "M4/outputs/metricas_transcripcion" / sub / "resumen.md"
    if ruta.exists():
        display(Markdown(ruta.read_text(encoding="utf-8")))

# Metricas de transcripcion (M4, tarea 5a)

Generado: 2026-11-01T03:26:21 · referencia principal: texto **visible** · manifiesto: `/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json`

CER y WER micro (errores totales / tamano total de la referencia). `--` = no aplica.

| Sistema | Nivel | Docs | CER visible | WER visible | CER completo | CER sin tildes | CER limpio (mismos docs) | Entidades leidas | Inventadas | Mal leidas |
|---|---|---|---|---|---|---|---|---|---|---|
| vlm | limpio | 15/15 | 0.000 | 0.001 | 0.000 | 0.000 | -- | 100.0 % | 0.0 % | 0.0 % |
| vlm | foto_realista | 15/15 | 0.003 | 0.004 | 0.000 | 0.003 | 0.000 | 100.0 % | 0.0 % | 0.0 % |

**Entidades leidas:** entidades del gold visibles en la imagen que aparecen tal cual en la transcripcion (no depende del NER). **Inventadas / mal leidas:** fraccion de las entidades predichas por el NER que no estan en el texto original (ver auditoria.py).

## Origen de las entidades predichas

| Sistema | Nivel | Predichas | Acierto | FP del NER | Texto extra | Mal leida | Inventada | Fuera de transcripcion |
|---|---|---|---|---|---|---|---|---|
| vlm | limpio | 32 | 29 | 3 | 0 | 0 | 0 | 0 |
| vlm | foto_realista | 32 | 29 | 3 | 0 | 0 | 0 | 0 |


> **SIMULADO.** Estas cifras salen de transcripciones simuladas con errores inyectados.
> Sirven para validar el modulo, no son resultados del sistema.

# Metricas de transcripcion (M4, tarea 5a)

Generado: 2026-11-01T03:26:18 · referencia principal: texto **visible** · manifiesto: `/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json`

CER y WER micro (errores totales / tamano total de la referencia). `--` = no aplica.

| Sistema | Nivel | Docs | CER visible | WER visible | CER completo | CER sin tildes | CER limpio (mismos docs) | Entidades leidas | Inventadas | Mal leidas |
|---|---|---|---|---|---|---|---|---|---|---|
| simulado | limpio | 15/15 | 0.135 | 0.171 | 0.135 | 0.131 | -- | 86.0 % | 19.0 % | 10.1 % |
| simulado | foto_realista | 15/15 | 0.194 | 0.407 | 0.196 | 0.188 | 0.135 | 73.7 % | 17.7 % | 19.0 % |
| simulado | manuscrito_foto | 15/15 | 0.202 | 0.376 | 0.202 | 0.200 | 0.135 | 80.7 % | 12.9 % | 10.0 % |
| simulado | sombra_severo | 15/15 | 0.265 | 0.503 | 0.359 | 0.260 | 0.135 | 58.2 % | 28.4 % | 13.4 % |
| simulado | rotacion_recorte_severo | 15/15 | 0.239 | 0.502 | 0.322 | 0.235 | 0.135 | 52.7 % | 25.4 % | 15.9 % |
| simulado | manuscrito_severo | 15/15 | 0.223 | 0.466 | 0.223 | 0.220 | 0.135 | 70.2 % | 16.8 % | 17.9 % |

**Entidades leidas:** entidades del gold visibles en la imagen que aparecen tal cual en la transcripcion (no depende del NER). **Inventadas / mal leidas:** fraccion de las entidades predichas por el NER que no estan en el texto original (ver auditoria.py).

## Origen de las entidades predichas

| Sistema | Nivel | Predichas | Acierto | FP del NER | Texto extra | Mal leida | Inventada | Fuera de transcripcion |
|---|---|---|---|---|---|---|---|---|
| simulado | limpio | 79 | 41 | 15 | 0 | 8 | 15 | 0 |
| simulado | foto_realista | 79 | 35 | 12 | 3 | 15 | 14 | 0 |
| simulado | manuscrito_foto | 70 | 41 | 9 | 4 | 7 | 9 | 0 |
| simulado | sombra_severo | 67 | 27 | 12 | 0 | 9 | 19 | 0 |
| simulado | rotacion_recorte_severo | 63 | 28 | 9 | 0 | 10 | 16 | 0 |
| simulado | manuscrito_severo | 95 | 36 | 20 | 6 | 17 | 16 | 0 |

## CER por dificultad de la caligrafia (niveles manuscritos)

| Sistema | Nivel | Dificultad | Docs | CER | WER |
|---|---|---|---|---|---|
| simulado | manuscrito_foto | dificil | 5 | 0.306 | 0.472 |
| simulado | manuscrito_foto | facil | 6 | 0.109 | 0.299 |
| simulado | manuscrito_foto | media | 4 | 0.202 | 0.353 |
| simulado | manuscrito_severo | dificil | 5 | 0.267 | 0.532 |
| simulado | manuscrito_severo | facil | 6 | 0.196 | 0.428 |
| simulado | manuscrito_severo | media | 4 | 0.203 | 0.429 |

La dificultad de cada caligrafia se etiqueto a ojo (ver M4/eval_visual/config.yaml).
